In [35]:
import random
import numpy as np
random.seed(42)
np.random.seed(42)

Generate Python code for read

In [4]:
from tensorflow.keras.datasets import mnist, cifar10
import numpy as np

(x_raw, y_raw), (x_test_raw, y_test)= mnist.load_data()

val_size = int(x_raw.shape[0] * 0.3)
rng = np.random.default_rng(42)  # seed for reproducibility
idx = np.arange(x_raw.shape[0])
rng.shuffle(idx)

val_idx = idx[:val_size]
train_idx = idx[val_size:]

(x_train_raw, y_train), (x_val_raw, y_val) = (x_raw[train_idx], y_raw[train_idx]), (x_raw[val_idx], y_raw[val_idx])



import matplotlib.pyplot as plt
import numpy as np
    
x_train, x_test, x_val= x_train_raw/255.0, x_test_raw/255.0, x_val_raw/255.0



In [8]:
import tensorflow as tf
from tensorflow.keras import layers, models
model = models.Sequential([
    layers.Input(shape = (28,28,1)),
    #1
    layers.Conv2D(filters=16, kernel_size = (3,3)),
    layers.BatchNormalization(),
    layers.Activation('relu'),

    #2
    layers.MaxPooling2D((2,2)),

    #3
    layers.Conv2D(filters=32, kernel_size=(3,3)),
    layers.BatchNormalization(),
    layers.Activation("relu"),

    #4
    layers.MaxPooling2D((2,2)),

    #5
    layers.Flatten(),
    layers.Dense(128, activation = "relu"),
    layers.Dropout(0.4),

    #6
    layers.Dense(10, activation = "softmax")
])

model.compile(optimizer = "adam", loss = "sparse_categorical_crossentropy", metrics = ["accuracy"]) #Is the loss and metrics correct?
fitted = model.fit(x_train, y_train, epochs = 5, batch_size = 64, validation_data = (x_val, y_val))

test_loss, test_acc = model.evaluate(x_test, y_test)
print("Test accuracy: ", test_acc)

Epoch 1/5
657/657 ━━━━━━━━━━━━━━━━━━━━ 12s 14ms/step - accuracy: 0.9186 - loss: 0.2649 - val_accuracy: 0.9771 - val_loss: 0.0786
Epoch 2/5
657/657 ━━━━━━━━━━━━━━━━━━━━ 9s 14ms/step - accuracy: 0.9708 - loss: 0.0939 - val_accuracy: 0.9830 - val_loss: 0.0575
Epoch 3/5
657/657 ━━━━━━━━━━━━━━━━━━━━ 9s 14ms/step - accuracy: 0.9786 - loss: 0.0689 - val_accuracy: 0.9820 - val_loss: 0.0585
Epoch 4/5
657/657 ━━━━━━━━━━━━━━━━━━━━ 10s 14ms/step - accuracy: 0.9819 - loss: 0.0598 - val_accuracy: 0.9861 - val_loss: 0.0469
Epoch 5/5
657/657 ━━━━━━━━━━━━━━━━━━━━ 10s 15ms/step - accuracy: 0.9837 - loss: 0.0508 - val_accuracy: 0.9871 - val_loss: 0.0449
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9877 - loss: 0.0365
Test accuracy:  0.9876999855041504


Brute-force black-box attack: One of the most straightforward ways to carry out a 
black-box attack is to use a greedy random search. Here, starting from a correctly 
classified image, one adds salt-and-pepper noise (randomly changing a few 
pixels) step-by-step, until the image is misclassified. More specifically, for a given 
step, after changing the image as just described, one keeps the modified image if 
the probability for the ground truth class 𝐶 is reduced after the modification. If not, 
the previous image is retained, and noise is added again, and so on. Implement 
and run this approach (visualization will follow in Step 5 below) over a few MNIST 
images. 

In [6]:
import random

random.seed(42)

import tensorflow as tf
from tensorflow.keras import layers, models

In [9]:
x_test_extra = x_test[..., None]     
x_train_extra = x_train[..., None]  
x_val_extra = x_val[..., None]
pred = 0
actual = np.inf
while pred!= actual:
    picked_index = random.randint(0, len(x_test_extra)-1)

    probs = model(x_test_extra[picked_index:picked_index+1]).numpy()[0]   # x_train[5:6] has shape (1,28,28,1)
    pred = np.argmax(probs)
    
    actual = y_test[picked_index]


In [11]:
n  = 10

x_old = x_test_extra[picked_index].copy()

def predict(model, image):
    probabilities = model(image, training = False).numpy()[0]
    pred = int(np.argmax(probabilities))
    return pred, probabilities

pred, probs = predict(model, x_test_extra[picked_index:picked_index+1])

old_prob = probs[actual]

step = 0
max_step = 500

pred_new = pred


while pred_new == actual and step < max_step:

    H, W = x_old.shape[0], x_old.shape[1]   # 28, 28
    x_coords = [random.randint(0, W-1) for _ in range(n)]
    y_coords = [random.randint(0, H-1) for _ in range(n)]


    salt_pepper = [random.randint(0, 1) for _ in range(n)]


    x_new = x_old.copy()

    for i in range(n):
        x_new[y_coords[i], x_coords[i], 0] = salt_pepper[i]
    
    pred, probs = predict(model, x_new[None, ...])

    if probs[actual] < old_prob:
        old_prob = probs[actual]
        x_old = x_new.copy()
        pred_new = pred
    else: 
        print("No improvement")

    step += 1
    print(step, "Pred: ", pred, "Actual: ", actual, "Prob: ", probs[pred], "Old prob: ", old_prob, "\n")



No improvement
1 Pred:  6 Actual:  6 Prob:  0.99977 Old prob:  0.9997286 

2 Pred:  6 Actual:  6 Prob:  0.9993611 Old prob:  0.9993611 

3 Pred:  6 Actual:  6 Prob:  0.9899306 Old prob:  0.9899306 

No improvement
4 Pred:  6 Actual:  6 Prob:  0.9969639 Old prob:  0.9899306 

5 Pred:  6 Actual:  6 Prob:  0.98731345 Old prob:  0.98731345 

No improvement
6 Pred:  6 Actual:  6 Prob:  0.99382716 Old prob:  0.98731345 

No improvement
7 Pred:  6 Actual:  6 Prob:  0.9905539 Old prob:  0.98731345 

No improvement
8 Pred:  6 Actual:  6 Prob:  0.9962463 Old prob:  0.98731345 

9 Pred:  6 Actual:  6 Prob:  0.9780651 Old prob:  0.9780651 

No improvement
10 Pred:  6 Actual:  6 Prob:  0.992687 Old prob:  0.9780651 

11 Pred:  6 Actual:  6 Prob:  0.87079877 Old prob:  0.87079877 

12 Pred:  6 Actual:  6 Prob:  0.870426 Old prob:  0.870426 

13 Pred:  6 Actual:  6 Prob:  0.81485707 Old prob:  0.81485707 

No improvement
14 Pred:  6 Actual:  6 Prob:  0.93734354 Old prob:  0.81485707 

15 Pred:  6 Act

Boundary attack
Start with target image T, with ground truth class label C_T and another image I that is classified (label C_I) as something else than the target class
Then tweak I towards T, making it resemble the target
If CNN still classifies that image as C_I -> Successful
Otherwise -> Step rejected, then a smaller step is tried 
Implement and run ovar a few MNIST images

In [36]:
def boundary_attack(ci_image, target_image, model, ci_predicted, n, step):
    #Tweak image ci towards T
    x_indx = np.array([random.randint(0, ci_image.shape[1]-1) for _ in range(n)])
    y_indx = np.array([random.randint(0, ci_image.shape[0]-1) for _ in range(n)])

    ci_image_new = ci_image.copy()

    ci_image_new[x_indx, y_indx, 0] = target_image[x_indx, y_indx, 0]

    #If CNN still classifies image as belonging to class C_I
    predicted, probs = predict(model, ci_image_new[None, ...])
    if predicted == ci_predicted:
        #Perturbation was successful
        print("Perturbation successful")
        return predicted, ci_image_new, ci_image, n, step
    elif step >= max_step:
        print("Max steps reached")
        return predicted, ci_image_new, ci_image, n, step
    else: 
        return boundary_attack(ci_image, target_image, model, ci_predicted, n//2, step+1)





images_num = 5

for i in images_num:
    target_found = False
    while target_found == False:
        index = random.randint(0, len(x_test_extra)-1)
        pred, probs = predict(model, x_test_extra[index:index+1])
        if pred == y_test[index]:
            target_found = True

            target_indx = index
            target_actual = y_test[index]
            target_probabilities = probs
            target_image = x_test_extra[target_indx]

    I_found = False
    while I_found == False:
        index = random.randint(0, len(x_test_extra)-1)
        pred, probs = predict(model, x_test_extra[index:index+1])
        if pred != target_actual:
            I_found = True

            ci = index
            ci_predicted = pred
            i_probabilities = probs
            ci_image = x_test_extra[index]


    number_of_pixels_edited = 1000
    step = 0
    max_step = 500

    plt.figure()
    plt.subplot(1,2,1)
    plt.imshow(x_test_extra[ci], cmap="gray")
    plt.title(f"Original: predicted = {ci_predicted}")
    plt.subplot(1,2,2)
    plt.imshow(x_new, cmap = "gray")
    plt.title(f"Modified: predicted = {pred}")
    plt.show()












"""
n  = 2000
n_limit = 1
step = 0
max_step = 500

pred_new = target_actual

#Make i into t

x_old = x_test_extra[ci].copy()
x_target = x_test_extra[target_indx].copy()



while pred_new != ci_predicted and step < max_step and n> n_limit:

    H, W = x_old.shape[0], x_old.shape[1]   # 28, 28
    x_coords = [random.randint(0, W-1) for _ in range(n)]
    y_coords = [random.randint(0, H-1) for _ in range(n)]


    x_new = x_old.copy()

    for i in range(n):
        x_new[y_coords[i], x_coords[i], 0] = x_target[y_coords[i], x_coords[i], 0]
    
    pred, probs = predict(model, x_new[None, ...])

    if pred == ci_predicted:
        print("No improvement")
        n = int(n/2)

    step += 1
    print(step, "Pred: ", pred, "Actual: ", actual, "Prob: ", probs[pred], "Old prob: ", old_prob, "\n")
"""


TypeError: 'int' object is not iterable